# 04 — Modeling

Models for Quora Question Pairs evaluated with log-loss (cross-entropy).

Baseline setup:
- Prior baseline (constant probability) as a sanity check
- Logistic Regression on concatenated TF-IDF as the first practical baseline


In [16]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd

%load_ext autoreload
%autoreload 2

def _find_project_root(start: Path) -> Path:
    candidates = [start] + list(start.parents)
    for p in candidates:
        if (p / "src").exists() and (p / "data").exists():
            return p
    return start.parent

PROJECT_ROOT = _find_project_root(Path.cwd())
SRC_PATH = PROJECT_ROOT / "src"
sys.path.insert(0, str(SRC_PATH))

from modeling import (
    make_split,
    baseline_prior,
    build_tfidf_concat,
    train_logreg,
    train_xgboost,
    train_lstm_siamese,
    train_bert_embeddings_logreg,
    train_hf_bert_finetune,
)

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [17]:
import importlib.util

is_colab = importlib.util.find_spec("google.colab") is not None

def _has(name: str) -> bool:
    return importlib.util.find_spec(name) is not None

missing = []
if not _has("xgboost"):
    missing.append("xgboost")
if not _has("sentence_transformers"):
    missing.append("sentence-transformers")
if not _has("tensorflow"):
    missing.append("tensorflow")
if not _has("torch"):
    missing.append("torch")
if not _has("transformers"):
    missing.append("transformers")

if is_colab and missing:
    import subprocess
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q"] + missing)

has_xgboost = _has("xgboost")
has_st = _has("sentence_transformers")
has_tf = _has("tensorflow")
has_torch = _has("torch")
has_transformers = _has("transformers")

is_colab, has_xgboost, has_st, has_tf, has_torch, has_transformers

(False, True, True, True, True, True)

In [18]:
SAMPLE_N = None

RUN_LOGREG = True
RUN_XGBOOST = has_xgboost
RUN_LSTM = has_tf
RUN_BERT_EMB = has_st
RUN_HF_BERT = False and has_torch and has_transformers

SAMPLE_N, RUN_LOGREG, RUN_XGBOOST, RUN_LSTM, RUN_BERT_EMB, RUN_HF_BERT

(None, True, True, True, True, False)

In [19]:
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"

pre_pq = PROCESSED_DIR / "quora_preprocessed.parquet"
pre_csv = PROCESSED_DIR / "quora_preprocessed.csv"

if pre_pq.exists():
    df = pd.read_parquet(pre_pq)
elif pre_csv.exists():
    df = pd.read_csv(pre_csv)
else:
    raise FileNotFoundError("Run 02_preprocessing.ipynb first to create quora_preprocessed.(parquet|csv)")

need_cols = ["q1_classic", "q2_classic", "q1_norm", "q2_norm", "is_duplicate"]
missing_cols = [c for c in need_cols if c not in df.columns]
if missing_cols:
    raise KeyError(missing_cols)

df = df.dropna(subset=need_cols).copy()
df["is_duplicate"] = df["is_duplicate"].astype(int)

if SAMPLE_N is not None and SAMPLE_N < len(df):
    df = df.sample(SAMPLE_N, random_state=42).reset_index(drop=True)

df.shape

(323432, 10)

In [20]:
y = df["is_duplicate"].to_numpy()
sp = make_split(len(df), y, test_size=0.2, random_state=42, stratify=True)
baseline_ll = baseline_prior(sp.y_train, sp.y_val)
baseline_ll

0.6585248871634892

In [21]:
if RUN_LOGREG:
    vec, X = build_tfidf_concat(df["q1_classic"], df["q2_classic"], max_features=200000, ngram_range=(1, 2), min_df=2)
    X_train = X[sp.idx_train]
    X_val = X[sp.idx_val]
    logreg_model, logreg_ll = train_logreg(X_train, sp.y_train, X_val, sp.y_val, C=2.0, max_iter=200, n_jobs=-1)
else:
    logreg_ll = None

logreg_ll

0.4642757094443013

In [22]:
if RUN_XGBOOST:
    pair_path = PROCESSED_DIR / "pair_similarity_features.csv"
    if not pair_path.exists():
        raise FileNotFoundError("Run 03_feature_engineering.ipynb first to create pair_similarity_features.csv")

    feat = pd.read_csv(pair_path)
    cols = [c for c in feat.columns if c not in ["id", "is_duplicate"]]
    if "is_duplicate" in feat.columns:
        y_num = feat["is_duplicate"].to_numpy(dtype=int)
    else:
        y_num = df["is_duplicate"].to_numpy(dtype=int)

    X_num = feat[cols].to_numpy(dtype=np.float32)

    m = min(len(X_num), len(df))
    X_num = X_num[:m]
    y_num = y_num[:m]

    sp2 = make_split(m, y_num, test_size=0.2, random_state=42, stratify=True)

    X_train = X_num[sp2.idx_train]
    X_val = X_num[sp2.idx_val]
    xgb_model, xgb_ll = train_xgboost(X_train, sp2.y_train, X_val, sp2.y_val, params=None)
else:
    xgb_ll = None

xgb_ll

0.3520642795655013

In [23]:
if RUN_LSTM:
    q1 = df["q1_norm"].astype(str).to_numpy()
    q2 = df["q2_norm"].astype(str).to_numpy()

    q1_train = q1[sp.idx_train]
    q2_train = q2[sp.idx_train]
    q1_val = q1[sp.idx_val]
    q2_val = q2[sp.idx_val]

    lstm_model, lstm_tok, lstm_ll = train_lstm_siamese(
        q1_train, q2_train, sp.y_train,
        q1_val, q2_val, sp.y_val,
        max_words=100000,
        max_len=40,
        emb_dim=128,
        rnn_units=64,
        epochs=2,
        batch_size=512,
        seed=42,
    )
else:
    lstm_ll = None

lstm_ll

C:\Users\Dmity\AppData\Roaming\Python\Python312\site-packages\keras\src\export\tf2onnx_lib.py:8: FutureWarning: In the future `np.object` will be defined as the corresponding NumPy scalar.
  if not hasattr(np, "object"):


0.39399403626017454

In [25]:
if RUN_BERT_EMB:
    bert_embed_model, bert_embed_ll = train_bert_embeddings_logreg(
        df["q1_norm"], df["q2_norm"], df["is_duplicate"].to_numpy(), sp.idx_train, sp.idx_val,
        model_name="sentence-transformers/all-MiniLM-L6-v2",
        batch_size=256,
    )
else:
    bert_embed_ll = None

bert_embed_ll

0.5656258716781156

In [26]:
if RUN_HF_BERT:
    q1 = df["q1_norm"].astype(str).to_numpy()
    q2 = df["q2_norm"].astype(str).to_numpy()

    q1_train = q1[sp.idx_train]
    q2_train = q2[sp.idx_train]
    q1_val = q1[sp.idx_val]
    q2_val = q2[sp.idx_val]

    hf_model, hf_tok, hf_ll = train_hf_bert_finetune(
        q1_train, q2_train, sp.y_train,
        q1_val, q2_val, sp.y_val,
        model_name="distilbert-base-uncased",
        max_len=96,
        batch_size=16,
        epochs=1,
        lr=2e-5,
        seed=42,
    )
else:
    hf_ll = None

hf_ll

In [27]:
rows = [
    ("baseline_prior", baseline_ll),
    ("baseline_logreg_tfidf_concat", logreg_ll),
    ("xgboost_pair_features", xgb_ll),
    ("lstm_siamese_gru", lstm_ll),
    ("bert_embeddings_logreg", bert_embed_ll),
    ("hf_bert_finetune", hf_ll),
]
rows = [(n, v) for n, v in rows if v is not None]
pd.DataFrame(rows, columns=["model", "log_loss"]).sort_values("log_loss")

,model,log_loss
2,xgboost_pair_features,0.352064
3,lstm_siamese_gru,0.393994
1,baseline_logreg_tfidf_concat,0.464276
4,bert_embeddings_logreg,0.565626
0,baseline_prior,0.658525
